# Go-term enrichment plotting 
Imported from R script GO-term_enrichment.R

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

df = pd.read_csv("C:/Users/Sebas/OneDrive/Dokument/Master courses/MASTER THESIS/R project-Master Thesis/go_enrichment_conflict_candidates_BP_MF.csv")

# Cap fold enrichment at 20 to prevent outliers dominating bubble sizes
df["fold_enrichment_capped"] = df["fold_enrichment"].clip(upper=20)

# Shared color scale across all plots
VMIN = 0
VMAX = df["neg_log10_p"].quantile(0.95)

MALE_COLOR    = "#6BAED6"
FEMALE_COLOR  = "#E07B8A"
NEUTRAL_COLOR = "#9E9E9E"

# color scales 

SCALE_SHARED    = [[0, "#F0F0F0"], [1, "#2CA25F"]]          
SCALE_MF        = [[0, "#F0F0F0"], [1, "#7B2D8B"]]         
SCALE_AT        = [[0, "#F0F0F0"], [1, "#9E7BB5"]]          
SCALE_MALE      = [[0, "#F0F0F0"], [1, MALE_COLOR]]        
SCALE_FEMALE    = [[0, "#F0F0F0"], [1, FEMALE_COLOR]]       

# here one row is one significant GO-term per test set, not transcripts. 
print(df.shape)
print(df["description_set"].value_counts())
print(f"neg_log10_p range: {df['neg_log10_p'].min():.1f} - {df['neg_log10_p'].max():.1f}")
print(f"Color scale cap (95th pct): {VMAX:.1f}")

(524, 15)
description_set
AllThree_all               175
AllThree_femaleCopies      101
MaleFemale_all              87
AllThree_maleCopies         64
MaleFemale_maleCopies       58
MaleFemale_femaleCopies     39
Name: count, dtype: int64
neg_log10_p range: 1.3 - 19.3
Color scale cap (95th pct): 9.7


Plot 1: shared BP term between Male + Female and All three 

In [21]:
# Get shared GO IDs between the two full-category BP comparisons
mf_ids_bp = set(df[(df["description_set"] == "MaleFemale_all") & (df["ontology"] == "BP")]["GO.ID"])
at_ids_bp = set(df[(df["description_set"] == "AllThree_all")   & (df["ontology"] == "BP")]["GO.ID"])
shared_bp_ids = mf_ids_bp & at_ids_bp

shared_bp = df[
    (df["GO.ID"].isin(shared_bp_ids)) &
    (df["description_set"].isin(["MaleFemale_all", "AllThree_all"])) &
    (df["ontology"] == "BP")
].copy()

# Order terms by significance in AllThree_all (more terms, more stable ranking)
term_order = (
    shared_bp[shared_bp["description_set"] == "AllThree_all"]
    .sort_values("weight01")
    .head(10)["GO.ID"]
    .tolist()
)

shared_bp = shared_bp[shared_bp["GO.ID"].isin(term_order)].copy()

# Truncate long term names
shared_bp["Term_short"] = shared_bp["Term"].str[:45]

# Map GO.ID to truncated term for y-axis ordering
id_to_term = shared_bp.drop_duplicates("GO.ID").set_index("GO.ID")["Term_short"].to_dict()
shared_bp["Term_short"] = shared_bp["GO.ID"].map(id_to_term)

# Y-axis order: most significant at top
y_order = [id_to_term[g] for g in term_order if g in id_to_term]

# X-axis labels
shared_bp["Category_label"] = shared_bp["description_set"].map({
    "MaleFemale_all": "Male + Female",
    "AllThree_all":   "All three"
})

# for consistent bubbleplot size distributions 
shared_bp["size_plot"] = np.sqrt(shared_bp["Significant"]) + 2

fig = px.scatter(
    shared_bp,
    x="Category_label",
    y="Term_short",
    size="size_plot",
    color="neg_log10_p",
    color_continuous_scale=SCALE_SHARED,
    range_color=[0, 5],
    size_max=25,
    category_orders={
        "Term_short": y_order,
        "Category_label": ["Male + Female", "All three"]
    },
    labels={
        "neg_log10_p": "-log10(p)",
        "Significant": "Observed genes",
        "Category_label": "",
        "Term_short": ""
    },
    title="Shared enriched BP terms across conflict-candidate categories",
    hover_data={"Significant": True, "size_plot": False}
)

fig.update_layout(
    width=1000,
    height=600,
    font=dict(size=12),
    plot_bgcolor="white",
    paper_bgcolor="white",
    coloraxis_colorbar=dict(title="-log10(p)"),
    xaxis=dict(
        side="top",
        range=[-0.5, 1.5]
    )
)

fig.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig.update_yaxes(showgrid=True, gridcolor="lightgrey")

fig.show()

Top BP terms for Male + Female families
All duplicates | Male-biasde copies | Female-biased copies

In [22]:
mf_bp = df[
    (df["category"] == "Male + Female") &
    (df["ontology"] == "BP")
].copy()

mf_bp["Direction_label"] = mf_bp["description_set"].map({
    "MaleFemale_all":          "All duplicates",
    "MaleFemale_maleCopies":   "Male-biased copies",
    "MaleFemale_femaleCopies": "Female-biased copies"
})

# Top 10 per direction independently, ordered by significance
top_terms_per_direction = (
    mf_bp.sort_values("weight01")
    .groupby("Direction_label")
    .head(10)
)

# Union of all selected GO IDs for filtering
top_go_ids = set(top_terms_per_direction["GO.ID"])

mf_bp_plot = mf_bp[mf_bp["GO.ID"].isin(top_go_ids)].copy()
mf_bp_plot["Term_short"] = mf_bp_plot["Term"].str[:120]

# Y-axis order: by significance in All duplicates, then others
all_order = (
    mf_bp_plot[mf_bp_plot["Direction_label"] == "All duplicates"]
    .sort_values("weight01")["Term_short"]
    .tolist()
)
male_only = (
    mf_bp_plot[
        (mf_bp_plot["Direction_label"] == "Male-biased copies") &
        (~mf_bp_plot["Term_short"].isin(all_order))
    ]
    .sort_values("weight01")["Term_short"]
    .tolist()
)
female_only = (
    mf_bp_plot[
        (mf_bp_plot["Direction_label"] == "Female-biased copies") &
        (~mf_bp_plot["Term_short"].isin(all_order + male_only))
    ]
    .sort_values("weight01")["Term_short"]
    .tolist()
)

y_order = all_order + male_only + female_only

# Local size columns, sqrt gradient for the 2-40 range
# sqrt(2)=1.4, sqrt(10)=3.2, sqrt(20)=4.5, sqrt(40)=6.3
mf_bp_plot["size_local"] = np.sqrt(mf_bp_plot["Significant"])

fig = px.scatter(
    mf_bp_plot,
    x="Direction_label",
    y="Term_short",
    size="size_local",
    color="neg_log10_p",
    color_continuous_scale=SCALE_MF,
    range_color=[0, 7],
    size_max=25,
    category_orders={
        "Term_short": y_order,
        "Direction_label": ["All duplicates", "Male-biased copies", "Female-biased copies"]
    },
    labels={
        "neg_log10_p": "-log10(p)",
        "Significant": "Observed genes",
        "Direction_label": "",
        "Term_short": ""
    },
    title="Top enriched BP terms: Male + Female families",
    hover_data={"Significant": True, "size_local": False}
)


fig.update_layout(
    width=1000,
    height=800,
    font=dict(size=12),
    title=dict(font=dict(size=13)),
    plot_bgcolor="white",
    paper_bgcolor="white",
    coloraxis_colorbar=dict(title="-log10(p)"),
    xaxis=dict(side="top"),
)

fig.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig.update_yaxes(showgrid=True, gridcolor="lightgrey", tickfont=dict(size=10))

fig.show()

Top enriched BP ters for All three

In [23]:
at_bp = df[
    (df["category"] == "All three") &
    (df["ontology"] == "BP")
].copy()

at_bp["Direction_label"] = at_bp["description_set"].map({
    "AllThree_all":          "All duplicates",
    "AllThree_maleCopies":   "Male-biased copies",
    "AllThree_femaleCopies": "Female-biased copies"
})

top_terms_per_direction = (
    at_bp.sort_values("weight01")
    .groupby("Direction_label")
    .head(10)
)

top_go_ids = set(top_terms_per_direction["GO.ID"])
at_bp_plot = at_bp[at_bp["GO.ID"].isin(top_go_ids)].copy()
at_bp_plot["Term_short"] = at_bp_plot["Term"].str[:120]

all_order = (
    at_bp_plot[at_bp_plot["Direction_label"] == "All duplicates"]
    .sort_values("weight01")["Term_short"]
    .tolist()
)
male_only = (
    at_bp_plot[
        (at_bp_plot["Direction_label"] == "Male-biased copies") &
        (~at_bp_plot["Term_short"].isin(all_order))
    ]
    .sort_values("weight01")["Term_short"]
    .tolist()
)
female_only = (
    at_bp_plot[
        (at_bp_plot["Direction_label"] == "Female-biased copies") &
        (~at_bp_plot["Term_short"].isin(all_order + male_only))
    ]
    .sort_values("weight01")["Term_short"]
    .tolist()
)

y_order = all_order + male_only + female_only
at_bp_plot["size_local"] = np.sqrt(at_bp_plot["Significant"])

fig = px.scatter(
    at_bp_plot,
    x="Direction_label",
    y="Term_short",
    size="size_local",
    color="neg_log10_p",
    color_continuous_scale=SCALE_AT,
    range_color=[0, 15],
    size_max=18,
    category_orders={
        "Term_short": y_order,
        "Direction_label": ["All duplicates", "Male-biased copies", "Female-biased copies"]
    },
    labels={
        "neg_log10_p": "-log10(p)",
        "Significant": "Observed genes",
        "Direction_label": "",
        "Term_short": ""
    },
    title="Top enriched BP terms: All three families",
    hover_data={"Significant": True, "size_local": False}
)

fig.update_layout(
    width=1000,
    height=900,
    font=dict(size=12),
    title=dict(font=dict(size=13)),
    plot_bgcolor="white",
    paper_bgcolor="white",
    coloraxis_colorbar=dict(title="-log10(p)"),
    xaxis=dict(side="top")
)

fig.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig.update_yaxes(showgrid=True, gridcolor="lightgrey", tickfont=dict(size=10))

fig.show()

MF directional plot for All three

MF: Male biased copies in All three

In [30]:
at_male_mf = df[
    (df["description_set"] == "AllThree_maleCopies") &
    (df["ontology"] == "MF")
].copy()

# Keep only terms unique to male copies (not in female)
at_female_mf_ids = set(df[
    (df["description_set"] == "AllThree_femaleCopies") &
    (df["ontology"] == "MF")
]["GO.ID"])

at_male_mf_unique = at_male_mf[~at_male_mf["GO.ID"].isin(at_female_mf_ids)].copy()
at_male_mf_unique["Term_short"] = at_male_mf_unique["Term"].str[:120]
at_male_mf_unique = at_male_mf_unique.sort_values("weight01").head(13)
y_order = at_male_mf_unique.sort_values("weight01", ascending=False)["Term_short"].tolist()
at_male_mf_unique["size_local"] = np.sqrt(at_male_mf_unique["Significant"])

fig = px.scatter(
    at_male_mf_unique,
    x="fold_enrichment_capped",
    y="Term_short",
    size="size_local",
    color="neg_log10_p",
    color_continuous_scale=SCALE_MALE,
    range_color=[0, 6],
    size_max=15,
    category_orders={"Term_short": y_order},
    labels={
        "neg_log10_p":           "-log10(p)",
        "Significant":           "Observed genes",
        "fold_enrichment_capped": "Fold enrichment",
        "Term_short":            ""
    },
    title="MF terms unique to male-biased copies: All three families",
    hover_data={"Significant": True, "size_local": False}
)

fig.update_layout(
    width=1000,
    height=600,
    font=dict(size=12),
    title=dict(font=dict(size=13)),
    plot_bgcolor="white",
    paper_bgcolor="white",
    coloraxis_colorbar=dict(title="-log10(p)"),
)

fig.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig.update_yaxes(showgrid=True, gridcolor="lightgrey", tickfont=dict(size=10))

fig.show()

MF: Female biased copies in All three

In [29]:
at_female_mf = df[
    (df["description_set"] == "AllThree_femaleCopies") &
    (df["ontology"] == "MF")
].copy()

at_male_mf_ids = set(df[
    (df["description_set"] == "AllThree_maleCopies") &
    (df["ontology"] == "MF")
]["GO.ID"])

at_female_mf_unique = at_female_mf[~at_female_mf["GO.ID"].isin(at_male_mf_ids)].copy()
at_female_mf_unique["Term_short"] = at_female_mf_unique["Term"].str[:120]
at_female_mf_unique = at_female_mf_unique.sort_values("weight01").head(16)
y_order = at_female_mf_unique.sort_values("weight01", ascending=False)["Term_short"].tolist()
at_female_mf_unique["size_local"] = np.sqrt(at_female_mf_unique["Significant"])

fig = px.scatter(
    at_female_mf_unique,
    x="fold_enrichment_capped",
    y="Term_short",
    size="size_local",
    color="neg_log10_p",
    color_continuous_scale=SCALE_FEMALE,
    range_color=[0, 7],
    size_max=15,
    category_orders={"Term_short": y_order},
    labels={
        "neg_log10_p":           "-log10(p)",
        "Significant":           "Observed genes",
        "fold_enrichment_capped": "Fold enrichment",
        "Term_short":            ""
    },
    title="MF terms unique to female-biased copies: All three families",
    hover_data={"Significant": True, "size_local": False}
)

fig.update_layout(
    width=1000,
    height=600,
    font=dict(size=12),
    title=dict(font=dict(size=13)),
    plot_bgcolor="white",
    paper_bgcolor="white",
    coloraxis_colorbar=dict(title="-log10(p)"),
)

fig.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig.update_yaxes(showgrid=True, gridcolor="lightgrey", tickfont=dict(size=10))

fig.show()